# BLaVe-CoT — Demo LoRA BLIP-2 (luồng VS Code + Colab extension)

Notebook này chạy bản demo nhỏ (~118 mẫu, 2 epoch, QLoRA 4-bit) trên GPU T4 Colab,
chỉ để **kiểm chứng pipeline chạy thông** trước khi train đầy đủ trên GPU riêng.

## Cách dùng với extension "Google Colab" của VS Code
1. Cài extension **Google Colab** (publisher: Google) từ Marketplace.
2. Đăng nhập Google trong VS Code (Command Palette → `Colab: Sign in`).
3. Mở file `.ipynb` này, click selector kernel góc phải trên → **Connect to a Colab runtime** → chọn **T4 GPU**.
4. Chạy lần lượt các cell bên dưới.

Lưu ý: kernel + filesystem chạy ở backend Colab, không phải máy local. File `demo_pack.zip` ở máy bạn cần upload qua `files.upload()` ở cell 2.

## 1. Kiểm tra GPU + clone repo

Sau khi đã connect runtime T4, chạy cell sau để xác nhận GPU và kéo code từ GitHub.

In [ ]:
import torch, os
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# Clone code (nếu chưa có)
if not os.path.isdir('blave_train'):
    !git clone https://github.com/HuynhKhoaIT/blave_train.git
%cd blave_train
!ls *.py

## 2. Cài lib + upload `demo_pack.zip`

`demo_pack.zip` (~50MB) đã được tạo trên máy bằng `make_demo.py`. Cell sau cài đủ lib và mở popup chọn file từ máy bạn để upload lên runtime Colab.

In [ ]:
!pip install -q -r requirements.txt

from google.colab import files
print("Chọn demo_pack.zip từ máy bạn...")
uploaded = files.upload()    # popup chọn demo_pack.zip
!unzip -q -o demo_pack.zip
!ls data/ && ls data/vizwiz/train | head -3

## 3. Verify config

Xác nhận PROFILE='demo' và đường dẫn khớp với data vừa unzip.

In [ ]:
!python config.py

## 4. Train

Lần đầu sẽ tải BLIP-2 base (~10GB) từ HuggingFace — mất ~3-5 phút. Sau đó train 2 epoch trên 118 mẫu, QLoRA 4-bit → khoảng 3-5 phút trên T4.

Nếu báo `target modules not found`: chạy `!python inspect_model.py` rồi sửa `LORA["target_modules"]` trong `config.py`.

In [ ]:
!python train.py

## 5. Inference — so sánh trước/sau fine-tune

Chọn 1 mẫu từ tập demo, chạy cả BLIP-2 gốc lẫn bản đã fine-tune để xem khác biệt.

In [ ]:
import json
sample = json.load(open('data/train_converted.json'))[0]
img = f"data/vizwiz/train/{sample['image']}"
print(f"Question: {sample['question']}")
print(f"Ground truth: {sample['answer']}\n")
!python infer.py --adapter ./blip2_lora_demo/final \
    --image "{img}" --question "{sample['question']}" --compare